# LangChain + MCP: Connecting Agents to External Tools

**Goal of this notebook:** learn the *client / integration* side of the Model
Context Protocol (MCP) — how to take existing MCP servers (or ones you write
yourself) and plug their tools straight into a LangChain / LangGraph agent,
instead of hand-writing a custom `@tool` for every integration.

> 📌 **Scope note:** this notebook focuses on **consuming** MCP servers.
> Building your **own** production MCP server is next week's topic. Here we
> build two tiny *toy* servers ourselves only so we have something to connect
> to and practice against — the real skill this week is the client side.

## What you'll do, step by step
1. MCP recap — servers, clients, tools, resources
2. Set up a **free** LLM (Groq) to power the agent
3. Install `langchain-mcp-adapters` and friends
4. Write a tiny MCP server and connect to it over **stdio**
5. Load MCP tools into LangChain and call one directly
6. Wire those tools into a **ReAct agent**
7. Write a second MCP server and connect over **HTTP** (streamable-http / SSE)
8. Use **`MultiServerMCPClient`** to combine tools from *multiple* servers into
   one agent
9. Wrap-up, gotchas, and what's next

Every code cell below was tested in a fresh environment before being put in
this notebook, so if something breaks for you it's almost always one of:
missing API key, a server not started yet, or a stale background process —
each of those is called out where it matters.


## 1. MCP recap

The **Model Context Protocol (MCP)** is an open standard (originally from
Anthropic) for connecting AI applications to external systems in a uniform
way — think "USB-C port for AI apps." Four concepts matter for the client
side:

| Concept | What it is |
|---|---|
| **MCP Server** | A small program that exposes capabilities — tools, resources, prompts — over a standard protocol. Could be local (a Python script) or remote (a hosted HTTP endpoint). |
| **MCP Client** | The piece that talks to a server: opens a connection (a *session*), lists what's available, and calls things. In this notebook, `langchain-mcp-adapters` *is* our client layer. |
| **Tools** | Functions the server exposes that an LLM can call — e.g. `add(a, b)`, `get_weather(city)`, `search_docs(query)`. This is what we'll load into LangChain as `BaseTool` objects. |
| **Resources** | Read-only data the server exposes (files, records, API responses) that can be pulled into context *without* being "called" like a tool — think of them like GET endpoints vs. POST actions. `langchain-mcp-adapters` can load these too (`client.get_resources(...)`), though this notebook focuses on tools, which is what agents use most. |

**Transports** — *how* the client and server actually talk to each other:

| Transport | When you'd use it | How it works |
|---|---|---|
| **stdio** | Local servers, e.g. a Python/Node script running on the same machine | Client spawns the server as a subprocess and talks over its stdin/stdout |
| **HTTP (streamable-http)** | Remote / hosted servers | Client makes HTTP requests to a URL like `https://.../mcp` |
| **SSE** | Older remote servers (being phased out in favor of streamable-http, but still common) | Server-Sent Events over HTTP |

The nice part: **once tools are loaded, your agent code doesn't care which
transport was used.** A tool from a local stdio math server and a tool from a
remote HTTP weather API look identical to the LLM.


## 2. Environment setup

We'll use **Groq** as our free LLM provider — it has a genuinely free tier
(no credit card), and its OpenAI-compatible API makes it a one-line swap with
`langchain-groq`. Gemini (`langchain-google-genai`) or Mistral
(`langchain-mistralai`) work as drop-in alternatives — see the note at the end
of this section if you'd rather use one of those.

**Get a free Groq API key:**
1. Go to https://console.groq.com/keys
2. Sign up (no card required) and create an API key
3. Paste it when prompted by the cell below (it won't be echoed to the
   notebook, and isn't saved to disk)


In [ ]:
# Install everything we need for this notebook.
# - langchain-mcp-adapters : the star of the show — MCP <-> LangChain bridge
# - mcp                    : the official MCP Python SDK (lets us WRITE toy servers too)
# - langgraph               : gives us create_react_agent for step 6
# - langchain-groq          : free LLM provider
%pip install -q langchain-mcp-adapters mcp langgraph langchain-groq langchain


In [ ]:
import os
import getpass

if not os.environ.get("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API key: ")

print("Groq key set:", bool(os.environ.get("GROQ_API_KEY")))


**Using Gemini or Mistral instead of Groq?** Swap the model block in every
agent cell below for one of these (everything else in the notebook — the MCP
parts — stays identical, since MCP tools work with *any* LangChain chat
model):

```python
# Gemini (free tier via Google AI Studio: https://aistudio.google.com/apikey)
%pip install -q langchain-google-genai
from langchain_google_genai import ChatGoogleGenerativeAI
model = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

# Mistral (free tier: https://console.mistral.ai/)
%pip install -q langchain-mistralai
from langchain_mistralai import ChatMistralAI
model = ChatMistralAI(model="mistral-large-latest")
```


### Sanity check: is the LLM working on its own?

Before bringing MCP into the picture at all, let's confirm the model itself
responds. If this cell fails, fix that first — everything after this depends
on it.


In [ ]:
from langchain_groq import ChatGroq

model = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)
response = model.invoke("Say hello in one short sentence.")
print(response.content)


## 3. `langchain-mcp-adapters` at a glance

This is the library that does the actual bridging. The pieces we'll use:

| Import | Purpose |
|---|---|
| `mcp.ClientSession`, `mcp.StdioServerParameters` | Low-level MCP SDK objects for opening a raw session |
| `mcp.client.stdio.stdio_client` | Spawns a local server as a subprocess (stdio transport) |
| `langchain_mcp_adapters.tools.load_mcp_tools(session)` | Converts an open MCP session's tools into a list of LangChain `BaseTool` objects |
| `langchain_mcp_adapters.client.MultiServerMCPClient` | Higher-level client: give it a dict of server configs (stdio *and/or* http *and/or* sse, mixed freely), call `.get_tools()`, get back one combined tool list |

You'll see two patterns below: the **manual session** pattern (more control,
good for understanding what's happening under the hood) and the
**`MultiServerMCPClient`** pattern (what you'll actually use day-to-day,
especially with more than one server).


## 4. Your first MCP server (stdio) + your first MCP client call

We'll write a tiny local "Math" server using `FastMCP` (a convenience wrapper
from the official `mcp` SDK for defining servers with plain Python functions
and decorators), save it to a file, then connect to it as a **subprocess**
over stdio — no ports, no networking, just stdin/stdout.


In [ ]:
%%writefile math_server.py
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("Math")

@mcp.tool()
def add(a: int, b: int) -> int:
    """Add two numbers."""
    return a + b

@mcp.tool()
def multiply(a: int, b: int) -> int:
    """Multiply two numbers."""
    return a * b

if __name__ == "__main__":
    mcp.run(transport="stdio")


Now let's connect to it. This is the "manual session" pattern: open a stdio
connection to the server process, wrap it in an `mcp.ClientSession`,
`initialize()` it (the MCP handshake), then hand the session to
`load_mcp_tools` to get back LangChain-ready tools.


In [ ]:
import asyncio
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client
from langchain_mcp_adapters.tools import load_mcp_tools

async def list_and_call_math_tools():
    server_params = StdioServerParameters(
        command="python3",
        args=["math_server.py"],   # the file we just wrote
    )
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()                 # MCP handshake
            tools = await load_mcp_tools(session)       # -> list[BaseTool]
            print("Tools exposed by the server:", [t.name for t in tools])
            for t in tools:
                print(f"  - {t.name}: {t.description}")

            # Call one directly, like any LangChain tool
            add_tool = next(t for t in tools if t.name == "add")
            result = await add_tool.ainvoke({"a": 3, "b": 5})
            print("\nadd(3, 5) ->", result)

await list_and_call_math_tools()


> **Note on notebooks:** `await` at the top level of a cell works out of the
> box in Jupyter (it runs your own event loop for you). If you're running this
> as a plain `.py` script instead, wrap calls in `asyncio.run(...)`.


## 5. Using MCP tools inside a ReAct agent

Loading tools by hand is useful for understanding the mechanics, but the
whole point is to hand a list of tools to an **agent** and let the LLM decide
when to call them. We'll use LangGraph's prebuilt `create_react_agent` — a
ready-made ReAct-style ("reason + act") loop: the model reasons about the
user's request, decides to call a tool, observes the result, and repeats
until it can answer.


In [ ]:
from langgraph.prebuilt import create_react_agent

async def run_math_agent(question: str):
    server_params = StdioServerParameters(command="python3", args=["math_server.py"])
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            tools = await load_mcp_tools(session)

            agent = create_react_agent(model, tools)
            result = await agent.ainvoke({"messages": question})

            # The final answer is the last message in the conversation
            print(result["messages"][-1].content)

await run_math_agent("What is (3 + 5) times 12? Use the tools available to you.")


Try changing the question above — e.g. `"What's 17 * 23, then add 100?"` —
and watch the agent chain multiple tool calls together on its own.

**Common gotcha:** the session (and therefore the tools) is only valid while
you're still *inside* the `async with` block — once the `stdio_client` /
`ClientSession` context managers exit, the subprocess is gone. That's why the
agent is created and invoked *inside* the `with` block above, not after it.


## 6. A second server, over HTTP (streamable-http)

Local stdio servers are great for local tools, but most real integrations
(a company API, a hosted database tool, a third-party SaaS) will be reachable
over the network instead. MCP's modern HTTP transport is called
**streamable-http** (the successor to plain SSE, though many servers still
support SSE for compatibility).

We'll write a tiny "Weather" server (data is mocked — no real API key needed)
and run it as a background web server on `localhost:8000`.


In [ ]:
%%writefile weather_server.py
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("Weather", host="127.0.0.1", port=8000)

@mcp.tool()
def get_weather(city: str) -> str:
    """Get the current weather for a city."""
    # Mocked data for practice -- swap this for a real weather API call
    # (e.g. Open-Meteo, which is free and needs no key) once you're comfortable.
    fake_data = {
        "ahmedabad": "34C, clear skies",
        "delhi": "31C, hazy",
        "mumbai": "29C, humid with light rain",
    }
    return fake_data.get(city.lower(), f"No data for {city}, assume 25C and sunny.")

if __name__ == "__main__":
    mcp.run(transport="streamable-http")


An MCP HTTP server needs to actually be **running** before a client can
connect to it — unlike the stdio server, which the client spawns on demand.
Start it as a background process from the notebook:


In [ ]:
import subprocess, time

weather_process = subprocess.Popen(
    ["python3", "weather_server.py"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)
time.sleep(2)  # give it a moment to bind the port
print("Weather server started, PID:", weather_process.pid)


Now connect to it — same `load_mcp_tools` pattern, just a different
transport. For HTTP servers, the MCP SDK gives us `streamablehttp_client`
instead of `stdio_client`.


In [ ]:
from mcp.client.streamable_http import streamablehttp_client

async def call_weather_tool(city: str):
    async with streamablehttp_client("http://127.0.0.1:8000/mcp") as (read, write, _):
        async with ClientSession(read, write) as session:
            await session.initialize()
            tools = await load_mcp_tools(session)
            print("Tools exposed by the server:", [t.name for t in tools])

            weather_tool = tools[0]
            result = await weather_tool.ainvoke({"city": city})
            print(f"\nWeather in {city}:", result)

await call_weather_tool("Ahmedabad")


## 7. `MultiServerMCPClient`: one client, many servers

Writing the raw session boilerplate for every server gets old fast —
especially once you want tools from *several* servers combined into a single
agent. `MultiServerMCPClient` solves this: describe every server (any mix of
`stdio`, `streamable_http`/`http`, or `sse`) in one config dict, call
`.get_tools()` once, and get back a single flat list of LangChain tools
spanning all of them.

### 7a. Single server via `MultiServerMCPClient`

First, the same weather server as above, but through the higher-level client
— notice there's no manual `ClientSession` at all now.


In [ ]:
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient({
    "weather": {
        "transport": "streamable_http",
        "url": "http://127.0.0.1:8000/mcp",
    },
})

weather_tools = await client.get_tools()
print("Tools:", [t.name for t in weather_tools])


### 7b. Multiple servers at once

This is the real payoff. One `MultiServerMCPClient`, math over stdio *and*
weather over HTTP, combined into a single tool list for one agent — the agent
doesn't know or care that they came from different servers over different
transports.


In [ ]:
client = MultiServerMCPClient({
    "math": {
        "command": "python3",
        "args": ["math_server.py"],
        "transport": "stdio",
    },
    "weather": {
        "transport": "streamable_http",
        "url": "http://127.0.0.1:8000/mcp",
    },
})

all_tools = await client.get_tools()
print("Combined tool list:", [t.name for t in all_tools])

agent = create_react_agent(model, all_tools)
result = await agent.ainvoke({
    "messages": "What's the weather in Mumbai, and separately, what is 45 * 6?"
})
print("\n" + result["messages"][-1].content)


Notice the agent correctly routes each part of the question to the right
server's tool — `get_weather` for Mumbai, `multiply` for the arithmetic — even
though from the agent's point of view they're just "tools," no different from
each other.

**If you're connecting to a server that needs auth** (many real hosted MCP
servers do), `streamable_http` and `sse` connections accept a `headers` field:

```python
client = MultiServerMCPClient({
    "some_saas": {
        "transport": "streamable_http",
        "url": "https://mcp.example.com/mcp",
        "headers": {"Authorization": "Bearer YOUR_TOKEN"},
    },
})
```


## 8. Cleanup

The weather server is still running as a background process. Stop it when
you're done experimenting (and re-run the "Start it as a background process"
cell in section 6 if you want to bring it back later).


In [ ]:
weather_process.terminate()
weather_process.wait()
print("Weather server stopped.")


## 9. Recap & what's next

You now know how to:
- ✅ Explain MCP's core pieces: servers, clients, tools, resources, transports
- ✅ Install and use `langchain-mcp-adapters`
- ✅ Connect to an MCP server over **stdio** (local subprocess) and over
  **HTTP/streamable-http** (networked)
- ✅ Load MCP tools as LangChain `BaseTool` objects with `load_mcp_tools`
- ✅ Wire MCP tools into a LangGraph **ReAct agent** with `create_react_agent`
- ✅ Combine tools from **multiple servers** with `MultiServerMCPClient`

### Ideas to practice further
- Swap the mocked `get_weather` for a real, free, keyless API like
  [Open-Meteo](https://open-meteo.com/) and watch the agent use real data.
- Try connecting to a public, third-party MCP server instead of one you
  wrote (e.g. search for "MCP servers directory" — there are open lists of
  community servers for GitHub, filesystems, databases, Slack, etc.).
- Add a **resource** to `math_server.py` (`@mcp.resource(...)`) and load it
  with `client.get_resources(...)` to see how resources differ from tools.
- Try the **SSE** transport against an older server (`"transport": "sse"`,
  `url` ending in `/sse`) to see the legacy pattern mentioned in section 1.

### Next week
Building your **own** MCP server from scratch (tools, resources, prompts) so
you can expose *your* systems to any MCP-compatible client — not just
LangChain agents.
